# [SITCOM-2111] Temperature Forecast Evaluation

* How accurate is the temperature forecast data provided by MeteoBlue?
* What is the different between forecast data and the actual data?

[SITCOM-2111]: https://ls.st/SITCOM-2111

## Setup

In [ ]:
day_obs_start = 20250501
day_obs_end = 20250515

In [ ]:
import logging
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from matplotlib.lines import Line2D

from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsForTime,
    getDayObsStartTime,
    makeEfdClient,
)
from astropy.time import Time

In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()

# Constants used in the notebook
ess_weather_station_sal_index = 301
m1m3_inside_cell_sal_index = 113
dome_inside_sal_index = 111

# Number of columns to consider - Maximum 336
N_COLS = 45

# CLT TimeZone Manual Offset
clt_tz_offset = -4  # hours

# Set global font size for labels, titles, and ticks
plt.rcParams.update(
    {
        "axes.grid": True,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "axes.formatter.useoffset": False,
        "axes.formatter.use_mathtext": False,
        "axes.formatter.limits": (-100, 100),
        "figure.figsize": (11, 6),
        "font.size": 12,
        "grid.color": "#b0b0b0",
        "grid.linestyle": ":",
        "grid.linewidth": 0.5,
        "grid.alpha": 0.75,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

## Helper Functions

In [ ]:
def query_ess_inside_dome(client, start_time, end_time):
    """
    Query the actual temperature inside the dome.
    """
    if not isinstance(start_time, Time):
        start_time = Time(start_time.to_pydatetime(), scale="utc")
    if not isinstance(end_time, Time):
        end_time = Time(end_time.to_pydatetime(), scale="utc")

    _df = getEfdData(
        client=client,
        topic="lsst.sal.ESS.temperature",
        columns=["temperatureItem0", "salIndex", "location"],
        begin=start_time,
        end=end_time,
    )

    # Select the data from the weather station using the salIndex
    mask = _df.salIndex == dome_inside_sal_index
    _df = _df[mask]

    # We do not need the salIndex anymore
    _df = _df.drop(columns=["salIndex"])

    # Get the rolling min/mean/max values for the temperature
    _df = _df.rename(columns={"temperatureItem0": "temperature"})
    _df = _df.resample("1min").agg({"temperature": ["min", "mean", "max"]})
    _df.columns = _df.columns.droplevel(0)

    return _df


def query_ess_weather_station(client, start_time, end_time):
    """
    Query the actual temperature from the weather station.
    """
    if not isinstance(start_time, Time):
        start_time = Time(start_time.to_pydatetime(), scale="utc")
    if not isinstance(end_time, Time):
        end_time = Time(end_time.to_pydatetime(), scale="utc")

    _df = getEfdData(
        client=client,
        topic="lsst.sal.ESS.temperature",
        columns=["temperatureItem0", "salIndex", "location"],
        begin=start_time,
        end=end_time,
    )

    # Select the data from the weather station using the salIndex
    mask = _df.salIndex == ess_weather_station_sal_index
    _df = _df[mask]

    # Get the rolling min/mean/max values for the temperature
    _df = _df.rename(columns={"temperatureItem0": "temperature"})
    _df = _df.resample("1min").agg({"temperature": ["min", "mean", "max"]})
    _df.columns = _df.columns.droplevel(0)

    return _df


def query_hourly_weather_forecast(client, day_obs_start, day_obs_end=None):
    """
    Query the forecast for a given day
    """
    if day_obs_end is None:
        day_obs_end = day_obs_start

    # The timedelta below helps including the first and last data point
    start_time = getDayObsStartTime(day_obs_start) - pd.to_timedelta("30min")
    end_time = getDayObsEndTime(day_obs_end) + pd.to_timedelta("30min")
    
    _df = getEfdData(
        client=client,
        topic="lsst.sal.WeatherForecast.hourlyTrend",
        columns=[f"timestamp{i}" for i in range(N_COLS)]
        + [f"temperature{i}" for i in range(N_COLS)]
        + [f"temperatureSpread{i}" for i in range(N_COLS)],
        begin=start_time,
        end=end_time,
    )

    for i in range(N_COLS):
        _df[f"timestamp{i}"] = pd.to_datetime(_df[f"timestamp{i}"], unit="s", utc=True)

    return _df


def unpack_hourly_weather_forecast(s):
    """
    Convert a Pandas Series into a DataFrame.

    Parameters
    ----------
    s : pd.Series
        A series containing timestampX, temperatureX,
        and temperatureSpreadX where X is a number from 0 to 335.
    """
    # Clean and reshape
    _df = s.rename_axis("key").reset_index(name="value")
    _df["field"] = _df["key"].str.extract(r"([a-zA-Z]+(?:Spread)?)")[0]
    _df["index"] = _df["key"].str.extract(r"(\d+)$")[0].astype("Int64")

    # Pivot to final table
    _df = _df.pivot(index="index", columns="field", values="value").reset_index(drop=True)
    _df["timestamp"] = pd.to_datetime(_df["timestamp"])
    _df = _df.where(pd.notnull(_df), pd.NA)
    _df.columns.name = None

    return _df


def unpack_hourly_weather_forecast_matching_timestamps(_df):
    """
    Use the published data to select the timestamps and temperatures 
    we want to include in the output dataframe.

    Parameters
    ----------
    _df : pd.DataFrame
        The weather forecast data. 
        The index must contain the timestamp of the published data. 
        It must have at least 40 columns for the timestamp,
        temperature, and temperatureSpread.
    """
    # Round published time to make it easier to select good data
    _df.index = _df.index.round("h")

    # Exclude rows published at times different from expected.
    _df = _df[_df.index.hour.isin([4, 16])]

    # Columns that starts with `timestamp`
    cols = [col for col in _df.columns if col.startswith("timestamp")]

    # Create empty dataframe
    new_df = None

    # Extract time range associated with the time between published times
    for published_time, row in _df[cols].iterrows():

        # Get the beginning of the data corresponding to this time range
        index_start = (row - published_time).abs().argmin()

        # Number of hours between published forecasts is always the same
        index_end = index_start + 12

        new_cols = (
            [f"timestamp{i}" for i in range(index_start, index_end)] + 
            [f"temperature{i}" for i in range(index_start, index_end)] + 
            [f"temperatureSpread{i}" for i in range(index_start, index_end)]
        )
        
        sub_df = _df[new_cols]
        sub_df = unpack_hourly_weather_forecast(sub_df.loc[published_time])
        sub_df["published_time"] = published_time

        if new_df is not None:
            new_df = pd.concat([new_df, sub_df])
        else:
            new_df = sub_df

    return new_df

## Analysis

### Hourly Forecast

The telemetry comes packed with 24 * 14 columns (336) for each telemetry value 
and each time we get new published data.  
Columns ending with 0 correspond to the forecast for the first hour after the publication time.  
Columns ending with 1 correspond to the forecast fot the second hour after the publication time.  
And so it goes up to 14 days.  

The forecast is published twice a day.   
Once near 4 UTC and once near 16h UTC.

In [ ]:
t_start = getDayObsStartTime(day_obs_start)
t_end = getDayObsEndTime(day_obs_end)

print(f" Query datasets between {t_start.isot} and {t_end.isot}")

In [ ]:
df_hourly_weather_forecast = query_hourly_weather_forecast(efd_client, day_obs_start, day_obs_end)
df_hourly_weather_forecast.head()

Assuming that the timestamps are all correct, let's do some further analysis to understand the data.  
First, let's compare two forecast rows that start at 00h UTC on that same day.  

In [ ]:
_df1 = unpack_hourly_weather_forecast(df_hourly_weather_forecast.iloc[0])
_df2 = unpack_hourly_weather_forecast(df_hourly_weather_forecast.iloc[1])

title = f"Forecast Comparison for {df_hourly_weather_forecast["timestamp0"].iloc[0]}"
fig, ax = plt.subplots(num=title, ncols=1, nrows=1)

ax.plot(_df1.timestamp, _df1.temperature, label=f"Published at {df_hourly_weather_forecast.index[0]}")
ax.plot(_df2.timestamp, _df2.temperature, label=f"Published at {df_hourly_weather_forecast.index[1]}")

ax.set_xlabel("Time [UTC]")
ax.set_ylabel("Temperature (ºC)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))

fig.suptitle(title)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

In [ ]:
df_forecast = unpack_hourly_weather_forecast_matching_timestamps(
    query_hourly_weather_forecast(efd_client, day_obs_start, day_obs_start+2)
)

df_weather_station = query_ess_weather_station(
    efd_client, df_forecast.timestamp.iloc[0], df_forecast.timestamp.iloc[-1]
)

df_inside_dome = query_ess_inside_dome(
    efd_client, df_forecast.timestamp.iloc[0], df_forecast.timestamp.iloc[-1]
)

In [ ]:
print("MeteoBlue hourly data-frame - head")
df_forecast.head()

In [ ]:
print("Weather station data-frame - head")
df_weather_station.head()

In [ ]:
print("Inside Dome data-frame - head")
df_inside_dome.head()

In [ ]:
title = f"MeteoBlue Reliability Study - {day_obs_start}"
tz_offset = pd.to_timedelta(f"{clt_tz_offset}h")

fig, ax = plt.subplots(num=title, ncols=1, nrows=1)

ax.plot(df_forecast.timestamp - tz_offset, df_forecast.temperature, "C0-", label="MeteoBlue Hourly Trend")
ax.plot(df_forecast.timestamp - tz_offset, df_forecast.temperature - df_forecast.temperatureSpread, "C0-", alpha=0.3)
ax.plot(df_forecast.timestamp - tz_offset, df_forecast.temperature + df_forecast.temperatureSpread, "C0-", alpha=0.3)
ax.plot(
    df_weather_station.index,
    df_weather_station["mean"],
    "C1-",
    label="Weather Station Temperature",
)
ax.plot(df_inside_dome.index, df_inside_dome["mean"], "C2-", label="Inside Dome Temperature")

ax.set_xlabel("Time [UTC]")
ax.set_ylabel("Temperature (ºC)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))

fig.suptitle(title)
fig.autofmt_xdate()
fig.tight_layout()

plt.savefig(f"{title}_t_offset.png")
plt.show()

## Hourly Forecast for multiple days

In [ ]:
time_start = getDayObsStartTime(day_obs_start)
time_end = getDayObsEndTime(day_obs_end)

print(f"Query data from {day_obs_start} to {day_obs_end}")

df_forecast = unpack_hourly_weather_forecast_matching_timestamps(
        query_hourly_weather_forecast(efd_client, day_obs_start, day_obs_end)
    )

df_weather_station = query_ess_weather_station(efd_client, time_start, time_end)

df_inside_dome = query_ess_inside_dome(efd_client, time_start, time_end)

In [ ]:
title = f"Hourly Forecast from {day_obs_start} to {day_obs_end}"
tz_offset = pd.to_timedelta(f"{clt_tz_offset}h")

fig, ax = plt.subplots(num=title)

# Plot forecast
ax.plot(df_forecast.timestamp - tz_offset , df_forecast.temperature, "C0-", label="Meteoblue Latests Forecast")
ax.plot(df_forecast.timestamp - tz_offset , df_forecast.temperature - df_forecast.temperatureSpread, "C0-", alpha=0.3)
ax.plot(df_forecast.timestamp - tz_offset , df_forecast.temperature + df_forecast.temperatureSpread, "C0-", alpha=0.3)

# Plot real data
ax.plot(df_weather_station["mean"], "C1-", label="Weather Station Temperature")
ax.plot(df_inside_dome["mean"], "C2-", label="Inside Dome Temperature")

# Plot every time the data was published
for pub_t in df_forecast["published_time"].unique():
    ax.axvline(x=pub_t, ls=":", alpha=0.5)

# Add it to the existing legend
legend_line = Line2D(
    [0], [0], color="gray", linestyle="--", label="Published Timestamp"
)

handles, labels = ax.get_legend_handles_labels()
handles.append(legend_line)
labels.append("Published Timestamp")

ax.legend(handles, labels, ncols=2)

# Cosmetics
t_start = getDayObsStartTime(day_obs_start).to_datetime()
t_end = getDayObsEndTime(day_obs_end).to_datetime()

ax.set_xlim(t_start, t_end)
ax.set_xlabel("Time [UTC]")
ax.set_ylabel("Temperature [deg C]")

fig.suptitle(title)
fig.tight_layout()
fig.autofmt_xdate()

plt.savefig(f"hourly_forecast_{day_obs_start}_to_{day_obs_end}.png")
plt.show()